# 노트북 정답 (N-1 ~ N-26)

빈칸 노트북 `노트북_빈칸.ipynb`의 `[N-k]` 번호와 1:1 대응합니다.
**스스로 채운 뒤에만** 여세요.

심화(선택) `BottleneckV1` 참고 구현도 아래에 있습니다 — 빈칸 노트북 3) BasicBlockV1 구간(N-6~N-13)의 답이
그대로 드러나므로 학습자 파일에서 이쪽으로 옮겼습니다.


In [ ]:
# =========================================================
# 정답 파일 — N-1 ~ N-30
# 빈칸 노트북(`노트북_빈칸.ipynb`)의 번호와 1:1 대응합니다.
# 먼저 스스로 채운 뒤에만 여세요.
#
# 2026-08-26 개정: conv/BN/ReLU가 한 줄에 겹쳐 있던 빈칸을 각각의 줄로 분리했습니다.
#                 그래서 번호가 26개 → 30개로 바뀌었습니다.
# =========================================================

# --- ShortcutZeroPad (cell 9) --------------------------------
# [N-1] x = x[:, :, ::self.stride, ::self.stride]
# [N-2] pad_channels = self.out_channels - self.in_channels
# [N-3] (x.size(0), pad_channels, x.size(2), x.size(3)),
# [N-4] return torch.cat([x, zeros], dim=1)

# --- ShortcutProjection (cell 13) ----------------------------
# [N-5] out = self.conv(x)
# [N-6] out = self.bn(out)

# --- BasicBlockV1 (cell 17) ----------------------------------
# [N-7]  self.shortcut = nn.Identity()
# [N-8]  self.shortcut = ShortcutProjection(in_channels, out_channels, stride=stride)
# [N-9]  self.shortcut = ShortcutZeroPad(in_channels, out_channels, stride=stride)
# [N-10] identity = self.shortcut(x)
# [N-11] out = self.conv1(x)
# [N-12] out = self.bn1(out)
# [N-13] out = self.relu(out)
# [N-14] out = self.conv2(out)
# [N-15] out = self.bn2(out)
# [N-16] out = out + identity
# [N-17] out = self.relu(out)

# --- ResNetV1.__init__ (cell 24) -----------------------------
# [N-18] stage_planes = [cifar_base_channels, cifar_base_channels * 2, cifar_base_channels * 4]
# [N-19] self.layer1 = self._make_layer(block, stage_planes[0], layers[0], stride=1)
# [N-20] self.layer2 = self._make_layer(block, stage_planes[1], layers[1], stride=2)
# [N-21] self.layer3 = self._make_layer(block, stage_planes[2], layers[2], stride=2)
# [N-22] self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

# --- ResNetV1._make_layer (cell 24) --------------------------
# [N-23] self.in_channels = planes * expansion
# [N-24] for _ in range(1, blocks):
# [N-25] out_channels = planes * expansion
# [N-26] self.in_channels = out_channels
# [N-27] for _ in range(1, blocks):
#
# N-23/N-25 참고: 위에서 `expansion = getattr(block, "expansion", 1)`로 지역변수를 만들어 두었으므로
#   `planes * expansion`과 `planes * block.expansion` 둘 다 정답입니다. 동작도 같습니다.

# --- resnet_cifar (cell 26) ----------------------------------
# [N-28] if (depth - 2) % 6 != 0:
# [N-29] n = (depth - 2) // 6
# [N-30] return ResNetV1(BasicBlockV1, [n, n, n], num_classes, stem="cifar", shortcut=shortcut)


## (심화·선택) BottleneckV1 참고 구현

ImageNet용 3층 블록. 쿡북 `[4) BottleneckV1]` 절과 함께 읽으세요.


In [ ]:

class BottleneckV1(nn.Module):
    """
    Figure 5 (right) bottleneck (ImageNet용에서 주로 사용).
    이 노트북에서는 심화(선택) 섹션으로만 다룹니다.
    """
    expansion = 4  # 최종 출력 채널 = planes * expansion

    def __init__(self, in_channels: int, planes: int, stride: int, shortcut: str = "projection"):
        super().__init__()
        out_channels = planes * self.expansion

        self.conv1 = nn.Conv2d(in_channels, planes, kernel_size=1, stride=1, padding=0, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)

        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.conv3 = nn.Conv2d(planes, out_channels, kernel_size=1, stride=1, padding=0, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channels)

        self.relu = nn.ReLU(inplace=True)

        # shortcut 선택
        if stride == 1 and in_channels == out_channels:
            self.shortcut = nn.Identity()
        else:
            # Bottleneck에서 zero_pad는 일반적으로 쓰지 않지만, 인터페이스 통일을 위해 허용
            if shortcut == "projection":
                self.shortcut = ShortcutProjection(in_channels, out_channels, stride=stride)
            elif shortcut == "zero_pad":
                self.shortcut = ShortcutZeroPad(in_channels, out_channels, stride=stride)
            else:
                raise ValueError(f"Unknown shortcut mode: {shortcut}")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = self.shortcut(x)

        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))

        out = out + identity
        out = self.relu(out)
        return out

